# 00 · Quickstart: from the spec to a first valid submission

Run top to bottom once `competition.toml` is set. A valid first submission proves the
submission format, ID alignment and metric direction before any optimisation.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
# os.environ["QRTC_CONFIG"] = "experiments/intraday_ridge/spec.toml"  # another spec
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60, "display.width", 160)
from qrtc import config, io, eda, features as F, cv, train, tracking, ensemble, postprocess, submit
from qrtc.metrics import get_metric
spec = config.load_spec()
spec

## 1. Load and sanity-check

In [ ]:
ds = io.load_dataset(spec)
print(ds.train.shape, ds.y.shape, ds.test.shape, "classes:", ds.classes)
ds.train.head()

In [ ]:
eda.split_structure(ds.train, ds.test, spec)

In [ ]:
eda.target_summary(ds.y, spec.task)

## 2. Baseline: LightGBM on raw features with the CV that mimics the train/test split

In [ ]:
feats = io.feature_columns(ds.train, spec)
X, X_test = F.to_category(ds.train[feats], ds.test[feats], F.auto_categoricals(ds.train[feats]))
splits = cv.make_splits(ds.train, ds.y_vector, spec, test=ds.test)
groups = ds.train[spec.date_col].to_numpy() if spec.date_col else None
weight = io.train_weight(ds, spec)
print(cv.describe_splits(splits))
model = "coxph" if spec.task == "survival" else "lgbm"
res = train.run_cv(spec, X, ds.y_vector, X_test, splits, model, weight=weight, groups=groups)

## 3. Save the run and write the submission (validated before writing)

In [ ]:
ids, test_ids = ds.train[spec.id_col].to_numpy(), ds.test[spec.id_col].to_numpy()
run_id = tracking.save_run(res, spec, f"quickstart-{model}", ids, test_ids, notes="first baseline")
sub = submit.build_submission(spec, test_ids, res.test_pred, ds.classes)
path = submit.write_submission(sub, spec, test_ids, run_id)
print(run_id, path)
sub.head()

## 4. Submit, then record the public score

`uv run qrtc record-lb <run_id> <public_score>` (or `tracking.record_lb(run_id, score)`).
Synthetic data only: `uv run qrtc score-hidden <path>` scores against the hidden test labels.

In [ ]:
# tracking.record_lb(run_id, 0.0)
tracking.leaderboard(competition=spec.name).head(10)